# Lesson 18 (follow-on): Receipts That Prove a *Human* Authorized the Action

Lesson 18 showed how an **agent** signs a receipt proving *it* took an action and that the record wasn't tampered with. This follow-on adds the other half: a receipt that proves a **named human approved the *exact* action** before it ran.

These are different properties:

| Receipt | Proves |
|---|---|
| Agent action receipt (Lesson 18) | the agent did what it claims, untampered |
| **Human approval receipt (this lesson)** | **a specific person said "yes" to *this exact action*** |

They are two independent signatures over partly-overlapping fields, joined by a content-derived reference — so neither the agent, the operator, nor the human can forge the other's half.

Same primitives as Lesson 18 — Ed25519 (`pynacl`) + RFC 8785 / JCS canonicalization (`jcs`) — and everything verifies **offline**, with no service to trust.


In [ ]:
# These are already the Lesson 18 dependencies — no new packages.
# %pip install pynacl jcs
import base64, hashlib
import jcs
from nacl.signing import SigningKey, VerifyKey
from nacl.exceptions import BadSignatureError

def b64u(b):      return base64.urlsafe_b64encode(b).rstrip(b"=").decode()
def b64u_dec(s):  return base64.urlsafe_b64decode(s + "=" * (-len(s) % 4))
def canon(obj):   return jcs.canonicalize(obj)            # RFC 8785 -> bytes
def sha256_hex(b): return hashlib.sha256(b).hexdigest()

## The exact action

The unit of approval is the **canonical action object** — not a vague label like "approve refund," but the precise, fully-specified action. Signing over the whole object (and deriving a digest from it) is what lets us later prove the human approved *this* and nothing else.

In [ ]:
action = {
    "action_type": "refund.issue",
    "params": {"order_id": "A-1029", "amount_usd": 4200, "to": "acct_88"},
    "policy_version": "refunds-v3",
}
print("action digest:", sha256_hex(canon(action)))

## The human approval receipt

In production the approver signs with a **WebAuthn platform authenticator** — a passkey, Face ID, or a security key. The signature is produced on the person's own device and is verifiable against the credential's published public key (and, for an auditor, its attestation chain). That hardware origin is what makes "a human said yes" *true* rather than a self-signed claim the operator could mint.

Here we stand in an Ed25519 key for the authenticator so the notebook runs offline — the sign/verify shape is identical.

> **The one rule that matters:** a verifier checks the receipt against the approver's **published / pinned** public key — *never* the key carried inside the receipt. Trusting the embedded key is the classic mistake; an attacker simply embeds their own.

In [ ]:
approver_sk = SigningKey.generate()
APPROVER_PUB = b64u(approver_sk.verify_key.encode())   # published out of band; pinned by the verifier

def human_approval(action, approver_id, approved_at_ms, sk=approver_sk):
    payload = {
        "typ": "human-approval",
        "approver_id": approver_id,
        "action": action,                       # the FULL canonical action, not just a digest
        "action_digest": sha256_hex(canon(action)),
        "approved_at_ms": approved_at_ms,
    }
    # approver_pub is for display only — verifiers MUST use the pinned key below.
    return {"payload": payload,
            "sig": b64u(sk.sign(canon(payload)).signature),
            "approver_pub": b64u(sk.verify_key.encode())}

def verify_human_approval(receipt, trusted_pub, action_being_executed):
    try:
        VerifyKey(b64u_dec(trusted_pub)).verify(canon(receipt["payload"]), b64u_dec(receipt["sig"]))
    except BadSignatureError:
        return (False, "signature invalid (forged or tampered)")
    p = receipt["payload"]
    if p["action"] != action_being_executed:
        return (False, "approval is not for the action being executed (confused deputy)")
    if p["action_digest"] != sha256_hex(canon(action_being_executed)):
        return (False, "action digest mismatch")
    return (True, f"approved by {p['approver_id']}")

In [ ]:
approval = human_approval(action, "alice@ops (WebAuthn)", 1_750_000_000_000)
print(verify_human_approval(approval, APPROVER_PUB, action))

## Joining the two receipts

The agent's action receipt (from Lesson 18) references the human approval by a **content-derived id** — `parent_approval_ref` = a hash of the approval's canonical bytes. Because it's derived from the bytes both sides recompute, neither side can forge the join.

We also enforce **one-time consumption**: an approval authorizes a single execution. Teaching replay-refusal matters as much as teaching signature-checking.

In [ ]:
agent_sk  = SigningKey.generate()
AGENT_PUB = b64u(agent_sk.verify_key.encode())

def approval_ref(receipt):                       # content-derived id of the approval
    return "ha_" + sha256_hex(canon(receipt["payload"]))[:32]

def agent_receipt(action, approval, executed_at_ms):
    payload = {
        "typ": "agent-action", "agent_id": "agent:refunds-bot",
        "action": action, "parent_approval_ref": approval_ref(approval),
        "executed_at_ms": executed_at_ms,
    }
    return {"payload": payload, "sig": b64u(agent_sk.sign(canon(payload)).signature), "agent_pub": AGENT_PUB}

_consumed = set()
def execute(action, approval, agent_rcpt):
    ok, why = verify_human_approval(approval, APPROVER_PUB, action)   # pinned key, not the receipt's
    if not ok: return (False, f"human approval: {why}")
    try:
        VerifyKey(b64u_dec(agent_rcpt["agent_pub"])).verify(canon(agent_rcpt["payload"]), b64u_dec(agent_rcpt["sig"]))
    except BadSignatureError:
        return (False, "agent receipt signature invalid")
    if agent_rcpt["payload"]["parent_approval_ref"] != approval_ref(approval):
        return (False, "agent receipt is not bound to this approval")
    ref = approval_ref(approval)
    if ref in _consumed: return (False, "approval already consumed (replay refused)")
    _consumed.add(ref)
    return (True, "executed")

receipt = agent_receipt(action, approval, 1_750_000_000_500)
print(execute(action, approval, receipt))

## What the binding catches

The whole point is that the approval is welded to the exact action. Each of these is refused — run and read the reason:

1. **Tampering** — the amount is changed after approval.
2. **Confused deputy** — a real approval for action A is replayed onto action B.
3. **Replay** — the same approval is used for a second execution.
4. **Forgery** — an attacker signs an approval with their own key.

In [ ]:
# 1. tamper: change the amount after approval
tampered = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("tamper        ->", verify_human_approval(approval, APPROVER_PUB, tampered))

# 2. confused deputy: valid approval for action A, used for action B
action_b = {**action, "action_type": "wire.send"}
print("confused-deputy ->", verify_human_approval(approval, APPROVER_PUB, action_b))

# 3. replay: reuse the same approval for a second execution
print("replay        ->", execute(action, approval, agent_receipt(action, approval, 1_750_000_001_000)))

# 4. forgery: attacker signs with their own key, checked against the pinned approver key
mallory_sk = SigningKey.generate()
forged = human_approval(action, "mallory", 1_750_000_002_000, sk=mallory_sk)
print("forged        ->", verify_human_approval(forged, APPROVER_PUB, action))

## What this proves — and what it doesn't

**Proves:** a specific, pinned human key approved *this exact action*; the approval can't be moved to another action, replayed, or forged; and anyone can check it offline without trusting the agent runtime or any server.

**Does not prove:** that the human understood what they approved (that's a UI / WYSIWYS concern — show the human the canonical action), or that the policy was actually enforced (the receipt records what was *claimed*). And the strength of "a human said yes" is only as strong as the authenticator: a real WebAuthn platform authenticator with user-verification is the property that makes it auditable — record *how* the human signed so a verifier can require a device-bound, user-verified approval for high-risk actions.

## Production references

- **WebAuthn / FIDO2** for the human-side signature, so the approval is rooted in a platform authenticator's attestation rather than a software key an operator controls.
- The receipt format and offline verifier in this lesson generalize; the **[EMILIA Protocol](https://github.com/emiliaprotocol/emilia-protocol)** publishes this human-authorization-receipt shape as IETF Internet-Drafts with an Apache-2.0 cross-language verifier (JavaScript / Python / Go) over the same RFC 8785 canonical bytes, including quorum (multi-approver) and revocation.

## Knowledge check

1. Why must the verifier use the approver's *pinned* public key instead of the one inside the receipt?
2. Why does the human sign the **full canonical action** rather than just an `action_digest`?
3. What stops a valid approval for a $4,200 refund from authorizing a $9,900 one?
